# Polars Demo — GHArchive on Ozone

Reads GitHub Archive data from Apache Ozone using Polars.
Polars reads directly from S3 via the `storage_options` API — no Spark needed.

In [ ]:
import os
import polars as pl

print("Polars version:", pl.__version__)

STORAGE = {
    "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "aws_endpoint_url": os.environ["AWS_S3_ENDPOINT"],
    "aws_allow_http": "true",
    "aws_virtual_hosted_style_request": "false",
}

## 1. List available data in raw bucket

In [ ]:
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    client_kwargs={"config": __import__("botocore").config.Config(s3={"addressing_style": "path"})},
)

partitions = fs.ls("raw/gh_archive/")
print("Top-level partitions:", partitions[:5])

## 2. Read a partition with Polars (lazy)

In [ ]:
# List one available file to read
all_files = fs.glob("raw/gh_archive/**/*.json.gz")
sample_file = "s3://" + all_files[0] if all_files else None
print("Sample file:", sample_file)

In [ ]:
# Read with Polars — lazy scan keeps memory usage low
lf = pl.scan_ndjson(sample_file, storage_options=STORAGE, infer_schema_length=1000)

# Show schema
print(lf.schema)

## 3. Aggregate: top event types

In [ ]:
top_events = (
    lf.select(pl.col("type"))
    .group_by("type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .limit(10)
    .collect()
)
print(top_events)

## 4. Filter and flatten nested fields

In [ ]:
push_events = (
    lf.filter(pl.col("type") == "PushEvent")
    .select([
        pl.col("id"),
        pl.col("created_at").cast(pl.Datetime),
        pl.col("actor").struct.field("login").alias("actor"),
        pl.col("repo").struct.field("name").alias("repo"),
    ])
    .collect()
)
print(f"{len(push_events)} PushEvents")
push_events.head(10)

## 5. Read from curated Iceberg data (Parquet files)

In [ ]:
parquet_files = fs.glob("curated/curated/github_events/data/**/*.parquet")
if parquet_files:
    curated_df = pl.read_parquet(
        ["s3://" + f for f in parquet_files[:5]],
        storage_options=STORAGE,
    )
    print(curated_df.shape)
    print(curated_df.head(10))
else:
    print("No curated data yet — run iceberg_demo.ipynb first.")